# 02 - Signal Research

Regime classifications and per-asset scores over the event log.
Uses `guardrail_lab.regime_analysis` for the transition matrix,
time-in-regime, and per-regime exposure, plus the raw
`asset_scored` events for the signal table.

**Offline-safe:** an empty event log prints a hint instead of
raising.


In [ ]:
# --- Guardrail Alpha notebook bootstrap (offline-safe) ---
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Return the first ancestor that contains python-lab/guardrail_lab."""
    for candidate in [start, *start.parents]:
        if (candidate / "python-lab" / "guardrail_lab").is_dir():
            return candidate
    return start


# In a notebook __file__ is undefined, so anchor on the current working dir.
REPO_ROOT = _find_repo_root(Path.cwd())
LAB_PATH = REPO_ROOT / "python-lab"
if str(LAB_PATH) not in sys.path:
    sys.path.insert(0, str(LAB_PATH))


def _first_existing(*candidates: Path):
    """Return the first path that exists, else None."""
    for path in candidates:
        if path.exists():
            return path
    return None


DATA_DIR = REPO_ROOT / "data"
CONFIG_DIR = REPO_ROOT / "configs"

# Prefer a real run; fall back to the seeded demo artifacts if present.
DB_PATH = _first_existing(
    DATA_DIR / "guardrail_alpha.db",
    DATA_DIR / "demo_guardrail_alpha.db",
)
RUN_REPORT_PATH = _first_existing(
    DATA_DIR / "run_report.json",
    DATA_DIR / "demo_run_report.json",
)
EXPERIMENTS_DIR = DATA_DIR / "experiments"
BACKTESTS_DIR = DATA_DIR / "backtests"
ELIGIBLE_ASSETS_PATH = CONFIG_DIR / "eligible_assets.bsc.json"
ASSET_CATEGORIES_PATH = CONFIG_DIR / "asset_categories.json"

NO_DATA_HINT = (
    "No data found under data/. Run the agent (paper/live) or seed a demo run "
    "first, e.g.  python3 -m guardrail_lab.seed  (from python-lab/), then "
    "re-run this notebook. The notebook is offline-safe and will not raise."
)

print("repo root :", REPO_ROOT)
print("event log :", DB_PATH if DB_PATH else "(none yet)")
print("run report:", RUN_REPORT_PATH if RUN_REPORT_PATH else "(none yet)")


In [ ]:
def render_table(rows, columns, title=None):
    """Print an aligned text table. rows: list[dict]; columns: list[(key,label)]."""
    if title:
        print(title)
        print("=" * len(title))
    if not rows:
        print("(no rows)")
        return
    labels = [label for _, label in columns]
    keys = [key for key, _ in columns]
    cells = [[("" if r.get(k) is None else str(r.get(k))) for k in keys] for r in rows]
    widths = [
        max(len(labels[i]), *(len(row[i]) for row in cells)) for i in range(len(keys))
    ]
    header = "  ".join(labels[i].ljust(widths[i]) for i in range(len(keys)))
    print(header)
    print("  ".join("-" * widths[i] for i in range(len(keys))))
    for row in cells:
        print("  ".join(row[i].ljust(widths[i]) for i in range(len(keys))))


## Load events


In [ ]:
from guardrail_lab.db import load_events, event_counts

events = load_events(str(DB_PATH)) if DB_PATH else []
if not events:
    print(NO_DATA_HINT)
else:
    print(f"Loaded {len(events)} events.")
    counts = event_counts(events)
    render_table(
        [{"event_type": k, "count": v} for k, v in counts.items()],
        [("event_type", "EVENT_TYPE"), ("count", "COUNT")],
        title="Event-type counts",
    )


## Regime analytics


In [ ]:
from guardrail_lab.regime_analysis import analyze_regimes

if events:
    analysis = analyze_regimes(events)

    print(f"Regimes observed: {analysis.transitions.regimes or '(none)'}")
    print(f"Total transitions: {analysis.transitions.total_transitions}\n")

    render_table(
        [
            {
                "regime": t.regime,
                "classifications": t.classifications,
                "fraction": f"{t.fraction:.2%}",
                "seconds": f"{t.seconds:.1f}",
            }
            for t in analysis.time_in_regime
        ],
        [
            ("regime", "REGIME"),
            ("classifications", "COUNT"),
            ("fraction", "SHARE"),
            ("seconds", "SECONDS"),
        ],
        title="Time in regime",
    )
    print()
    render_table(
        [
            {
                "regime": e.regime,
                "orders": e.order_count,
                "avg_order_usd": f"{e.avg_order_usd:,.2f}",
                "exposure_x": f"{e.exposure_multiplier:.3f}",
            }
            for e in analysis.exposure
        ],
        [
            ("regime", "REGIME"),
            ("orders", "ORDERS"),
            ("avg_order_usd", "AVG_ORDER_USD"),
            ("exposure_x", "EXPOSURE_x"),
        ],
        title="Exposure multiplier by regime",
    )
else:
    print(NO_DATA_HINT)


## Asset scores


In [ ]:
# asset_scored events: {"symbol": ..., "score": ...}
def _to_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

if events:
    score_rows = []
    for ev in events:
        if ev.get("event_type") != "asset_scored":
            continue
        payload = ev.get("payload") or {}
        score_rows.append(
            {
                "timestamp": ev.get("timestamp", ""),
                "symbol": payload.get("symbol", "?"),
                "score": _to_float(payload.get("score")),
            }
        )

    # Aggregate: mean score + count per symbol, ranked.
    agg = {}
    for r in score_rows:
        sym = r["symbol"]
        slot = agg.setdefault(sym, {"symbol": sym, "n": 0, "sum": 0.0, "last": None})
        slot["n"] += 1
        if r["score"] is not None:
            slot["sum"] += r["score"]
            slot["last"] = r["score"]

    ranked = sorted(
        (
            {
                "symbol": s["symbol"],
                "times_scored": s["n"],
                "mean_score": f"{(s['sum'] / s['n']):.3f}" if s["n"] else "n/a",
                "last_score": f"{s['last']:.3f}" if s["last"] is not None else "n/a",
            }
            for s in agg.values()
        ),
        key=lambda x: x["mean_score"],
        reverse=True,
    )

    if ranked:
        render_table(
            ranked,
            [
                ("symbol", "SYMBOL"),
                ("times_scored", "TIMES_SCORED"),
                ("mean_score", "MEAN_SCORE"),
                ("last_score", "LAST_SCORE"),
            ],
            title="Asset scores (ranked by mean score)",
        )
    else:
        print("No asset_scored events in the log yet.")
else:
    print(NO_DATA_HINT)


## Notes

* The exposure multiplier (`EXPOSURE_x`) shows how much larger or
  smaller proposed orders were during each regime versus the
  overall mean order size (1.0 = baseline).
* Higher mean scores indicate assets the signal stack favoured most
  consistently across cycles.
